# Feedforward Neural Network on MNIST

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

## Datasets and Data Loading
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Visualization
import matplotlib.pyplot as plt

## Numerical operations
import numpy as np

## Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

## Importing MNIST dataset

In [ ]:
## Transform images into tensors and normalize them
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

## Download and load the training dataset
train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

## Download and load the test dataset
test_dataset = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

## Use Dataloader to create batches of data to speed up training
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
## Checking to make sure data is loaded correctly
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

image, label = train_dataset[0]

print(image.shape)
print(label)

## Creating the Neural Network

In [ ]:
class FeedForwardNN(nn.Module):

    def __init__(self):
        super().__init__()

        ## 3 fully connected linear layers
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):

        ## Flatten the 28x28 tensor into a vector of size 784
        x = x.view(-1, 28 * 28)

        ## Use relu to introduce non-linearity to the model
        x = self.fc1(x)
        x = torch.relu(x)

        x = self.fc2(x)
        x = torch.relu(x)

        x = self.fc3(x)

        return x

## Create an instance of the model
model = FeedForwardNN().to(device)        

## Define the Loss Function

In [ ]:
## CrossEntropyLoss applies both softmax and cross-entropy loss. Softmax converts the outputs into proabilities, and 
## cross-entropy loss measures the difference between the predicted probabilities and the true labels.
criterion = nn.CrossEntropyLoss()

## Use an Optimizer

In [ ]:
## Optimizer applies the gradient to the model's weights to minimize the loss function.
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Train the Model

In [ ]:
num_epochs = 10

## 10 epochs of training
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        ## Forward pass
        outputs = model(images)

        ## Compute the loss
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        ## Zero gradients
        optimizer.zero_grad()

        ## Backward pass
        loss.backward()

        ## Update weights
        optimizer.step()

    average_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {average_loss:.4f}")    

## Validating the Model

In [ ]:
model.eval()

## Gradients are not needed since the model is not training anymore
with torch.no_grad():
    correct = 0
    total = 0

    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        ## Forward pass
        outputs = model(images)

        ## Get the predicted class with the highest probability
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")

## Visualizing the Results

In [ ]:
# Put the model in evaluation mode
model.eval()

# Get one batch of test images
images, labels = next(iter(test_loader))

# Move to device
images = images.to(device)
labels = labels.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(images)
    _, predictions = torch.max(outputs, 1)

# Move everything back to CPU for plotting
images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

# Plot the first 10 images
plt.figure(figsize=(12, 5))

for i in range(10):
    color = "green" if predictions[i] == labels[i] else "red"
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"Pred: {predictions[i]}\nTrue: {labels[i]}", color=color)
    plt.axis("off")

plt.tight_layout()
plt.show()